In [1]:
import sys
sys.path.insert(0, '/home/ethantu/workspace/good-vibrations/src3')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from vibrations_pipeline import _normalize_fft

In [2]:
SAMPLES_DIR = '/home/ethantu/workspace/good-vibrations/data/samples'

In [3]:
# One row per sample dir under SAMPLES_DIR, built from each sample's metadata.jsonl.
# fft_path points at the raw (unprocessed, complex) per-laser FFT shifts -- shape
# (1, n_lasers, n_freqs, 2), last dim = (x, y) shift direction -- loaded lazily by
# plot_fft_xy_magnitude rather than eagerly here, since each file is a few MB.
# com is the downsampled (mask-space) center of mass, matching the rest of the notebook.

def load_samples_df(samples_dir=SAMPLES_DIR):
    rows = []
    for sample_dir in sorted(Path(samples_dir).iterdir()):
        meta_path = sample_dir / 'metadata.jsonl'
        fft_path = sample_dir / 'inputs' / '03_fft_shifts.npz'
        if not meta_path.exists() or not fft_path.exists():
            continue
        meta = {}
        for line in meta_path.read_text().strip().splitlines():
            meta |= json.loads(line)
        com = meta.get('downsampled_com')
        rows.append({
            'sample_id':    int(meta['sample_id']),
            'output_id':    meta.get('output_id'),
            'speaker':      meta.get('speaker'),
            'n_objects':    meta.get('n_objects'),
            'box':          meta.get('box'),
            'object':       meta.get('object'),
            'is_empty_box': meta.get('is_empty_box'),
            'com_x':        com[0] if com is not None else None,
            'com_y':        com[1] if com is not None else None,
            'fft_path':     fft_path,
        })
    return pd.DataFrame(rows)

samples_df = load_samples_df()
samples_df

,sample_id,output_id,speaker,n_objects,box,object,is_empty_box,com_x,com_y,fft_path
0,0,000000,1,1,metal,cube,False,10.846068,10.192947,/home/ethantu/workspace/good-vibrations/data/s...
1,1,000000,2,1,metal,cube,False,10.846068,10.192947,/home/ethantu/workspace/good-vibrations/data/s...
2,2,000000,3,1,metal,cube,False,10.846068,10.192947,/home/ethantu/workspace/good-vibrations/data/s...
3,3,000000,4,1,metal,cube,False,10.846068,10.192947,/home/ethantu/workspace/good-vibrations/data/s...
4,4,000000,5,1,metal,cube,False,10.846068,10.192947,/home/ethantu/workspace/good-vibrations/data/s...
...,...,...,...,...,...,...,...,...,...,...
683,683,000085,4,0,metal,,True,-1.000000,-1.000000,/home/ethantu/workspace/good-vibrations/data/s...
684,684,000085,5,0,metal,,True,-1.000000,-1.000000,/home/ethantu/workspace/good-vibrations/data/s...
685,685,000085,6,0,metal,,True,-1.000000,-1.000000,/home/ethantu/workspace/good-vibrations/data/s...
686,686,000085,7,0,metal,,True,-1.000000,-1.000000,/home/ethantu/workspace/good-vibrations/data/s...


In [4]:
def _match(series, value):
    """value may be a single scalar or a list/tuple/set of allowed values."""
    values = [value] if not isinstance(value, (list, tuple, set)) else list(value)
    return series.isin(values)

def _zero_pad(value, width=6):
    """output_id is stored zero-padded (e.g. '000083'); accept plain ints too."""
    to_str = lambda v: f'{v:0{width}d}' if isinstance(v, int) else v
    return [to_str(v) for v in value] if isinstance(value, (list, tuple, set)) else to_str(value)

def filter_samples_df(df, sample_id=None, speaker=None, n_objects=None, box=None,
                      output_id=None, object=None, max_samples=None):
    """Each of sample_id/speaker/n_objects/box/output_id/object accepts either a single
    value or a list of values; rows matching any of them are kept."""
    df = df.copy()
    if sample_id is not None:  df = df[_match(df['sample_id'], sample_id)]
    if speaker is not None:    df = df[_match(df['speaker'], speaker)]
    if n_objects is not None:  df = df[_match(df['n_objects'], n_objects)]
    if box is not None:        df = df[_match(df['box'], box)]
    if output_id is not None:  df = df[_match(df['output_id'], _zero_pad(output_id))]
    if object is not None:     df = df[_match(df['object'], object)]
    if max_samples is not None:
        df = df.iloc[:max_samples]
    return df

In [5]:
XY_LABELS = ['x', 'y']

def compute_fft_magnitude(fft_path, laser_index=None, xy_index=None, normalize=True):
    """
    Core computation: load a sample's raw complex FFT (1, n_lasers, n_freqs, 2 -- last
    dim = x/y shift direction) and reduce it to a 1D magnitude spectrum (n_freqs,).

    laser_index: None -> average magnitude over all lasers; int -> that laser only.
    xy_index:    None -> average magnitude over x and y; 0 -> x only; 1 -> y only.
    normalize:   apply the same std-sample normalization used when building the dataset
                 (via vibrations_pipeline._normalize_fft) before reducing.

    Returns (freqs, magnitude), both 1D arrays of shape (n_freqs,).
    """
    with np.load(fft_path) as data:
        fft, freqs = data['fft'], data['freqs']  # fft: (1, L, F, 2) complex

    mag = np.abs(fft.astype(np.complex128)).astype(np.float32)  # (1, L, F, 2)
    if normalize:
        mag = _normalize_fft(mag, normalize_mode='std-sample')  # matches dataset preprocessing

    mag = mag[0]  # (L, F, 2)
    mag = mag[laser_index] if laser_index is not None else mag.mean(axis=0)  # (F, 2)
    mag = mag[:, xy_index] if xy_index is not None else mag.mean(axis=-1)    # (F,)
    return freqs, mag

In [6]:
def plot_fft_xy_magnitude(df, sample_id=None, speaker=None, n_objects=None, box=None,
                          output_id=None, object=None, laser_index=None, xy_index=None,
                          color_by='sample_id', normalize=True, log_y=False,
                          title=None, max_samples=None):
    """
    Plot each matching sample's FFT magnitude spectrum (see compute_fft_magnitude) as
    its own line on one figure. Filter kwargs accept a single value or a list of values
    (see filter_samples_df).

    color_by: df column to color lines by (e.g. 'sample_id', 'output_id', 'speaker') --
              lines sharing a value get the same color.
    """
    sdf = filter_samples_df(df, sample_id=sample_id, speaker=speaker, n_objects=n_objects,
                            box=box, output_id=output_id, object=object, max_samples=max_samples)
    if sdf.empty:
        print('No samples match the filter.')
        return

    palette = px.colors.qualitative.Plotly
    color_vals = sdf[color_by].unique()
    color_map = {v: palette[i % len(palette)] for i, v in enumerate(color_vals)}

    fig = go.Figure()
    for _, row in sdf.iterrows():
        sid = int(row['sample_id'])
        freqs, mag = compute_fft_magnitude(row['fft_path'], laser_index, xy_index, normalize)

        # empty box (n_objects == 0) stores com as the sentinel (-1, -1), not a real position
        com_str = (f"({row['com_x']:.1f}, {row['com_y']:.1f})"
                  if pd.notna(row.get('com_x')) and row.get('n_objects') != 0 else 'n/a')

        legend_label = f"id={sid} out={row.get('output_id')}"
        hover_label = (f"id={sid}<br>"
                      f"out={row.get('output_id')} spk={row.get('speaker')} "
                      f"obj={row.get('object')} n_obj={row.get('n_objects')} com={com_str}")

        fig.add_trace(go.Scatter(
            x=freqs, y=mag, mode='lines', name=legend_label, opacity=0.7,
            line=dict(color=color_map[row[color_by]]),
            hovertemplate=f'{hover_label}<br>freq=%{{x:.1f}} Hz<br>magnitude=%{{y:.4g}}<extra></extra>',
        ))

    laser_desc = 'mean over lasers' if laser_index is None else f'laser={laser_index}'
    xy_desc = 'mean over x/y' if xy_index is None else XY_LABELS[xy_index]
    fig.update_layout(title=title or f'FFT magnitude ({laser_desc}, {xy_desc}, colored by {color_by}, normalize={normalize})',
                      xaxis_title='freq (Hz)', yaxis_title='magnitude',
                      yaxis_type='log' if log_y else 'linear', height=450)
    fig.show()

In [7]:
# default: average magnitude over all 100 lasers and both x/y directions
plot_fft_xy_magnitude(samples_df, n_objects=0, color_by='output_id')

In [11]:
plot_fft_xy_magnitude(samples_df, speaker=1, output_id=[83, 84, 85, 0])

In [9]:
# Empty-box FFT magnitude, one subplot per speaker, overlaying the 3 output_ids
# (empty-box samplings) on top of each other within each subplot.

def plot_fft_xy_magnitude_by_speaker(df, speaker=None, output_id=None,
                                     laser_index=None, xy_index=None,
                                     color_by='output_id', normalize=True,
                                     log_y=False, ncols=1):
    sdf = filter_samples_df(df, speaker=speaker, output_id=output_id)
    if sdf.empty:
        print('No empty-box samples match the filter.')
        return

    speakers = sorted(sdf['speaker'].unique())
    color_vals = sdf[color_by].unique()
    palette = px.colors.qualitative.Plotly
    color_map = {v: palette[i % len(palette)] for i, v in enumerate(color_vals)}

    nrows = (len(speakers) + ncols - 1) // ncols
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f'speaker={s}' for s in speakers])

    for i, spk in enumerate(speakers):
        r, c = divmod(i, ncols)
        for _, row in sdf[sdf['speaker'] == spk].iterrows():
            sid = int(row['sample_id'])
            freqs, mag = compute_fft_magnitude(row['fft_path'], laser_index, xy_index, normalize)

            legend_label = f"id={sid} out={row.get('output_id')}"
            hover_label = f"id={sid}<br>out={row.get('output_id')} spk={spk}"

            fig.add_trace(go.Scatter(
                x=freqs, y=mag, mode='lines', name=legend_label, opacity=0.7,
                legendgroup=str(row[color_by]), showlegend=(i == 0),
                line=dict(color=color_map[row[color_by]]),
                hovertemplate=f'{hover_label}<br>freq=%{{x:.1f}} Hz<br>magnitude=%{{y:.4g}}<extra></extra>',
            ), row=r + 1, col=c + 1)

    laser_desc = 'mean over lasers' if laser_index is None else f'laser={laser_index}'
    xy_desc = 'mean over x/y' if xy_index is None else XY_LABELS[xy_index]
    fig.update_layout(title=f'Empty-box FFT magnitude by speaker ({laser_desc}, {xy_desc}, colored by {color_by}, normalize={normalize})',
                      height=300 * nrows, showlegend=True)
    fig.update_xaxes(title_text='freq (Hz)')
    fig.update_yaxes(title_text='magnitude', type='log' if log_y else 'linear')
    fig.show()

plot_fft_xy_magnitude_by_speaker(samples_df, speaker=[1, 2], output_id=[83, 84, 85, 0])